# Compiling Algol (semi-detached) systems 
## from Malkov 2020

based on 
### "Semidetached double-lined eclipsing binaries: Stellar parameters and rare classes "
Malkov 2020
 https://ui.adsabs.harvard.edu/abs/2020MNRAS.491.5489M/abstract



*** 
From Malkov (2020MNRAS.491.5489M): 

The well-known Catalogue of Algol-Type Binary Stars \cite{2004A&A...417..263B} lists 411 Semi detached stars, however, the majority of those stars are drawn from the \cite{1980AcA....30..501B} and \cite{2004yCat.5124....0S} catalogues which only contain approximate data. Lastly \cite{2004yCat.5115....0S} provide parameters for 96 semi-detached binaries.

\cite{2020MNRAS.491.5489M} makes a new comprehensive list of semi-detached binaries with reliable absolute parameters containing 119 semidetached double-lined eclipsing binaries containing the orbital parameters and physical parameters of the components.



In [9]:
import numpy as np
import pandas as pd
import h5py
import json
import sys

from astroquery.vizier import Vizier

# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
import os, sys
from pathlib import Path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR

## Helper functions to build the h5 table in consistent format

In [10]:
# ---- helper functions ----
columns = [
    "System Name", "RA", "Dec", "Period", "Eccentricity",
    "M1","M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function",
    "Type1", "Type2", "Detection Method", "Reference", "Notes"
]
df = pd.DataFrame(columns=columns)

def add_observation(df, system_name,
                    ra, dec, period, ecc,
                    m1, m1_sin3i, m2, m2_sin3i, q, mass_func,
                    type1, type2, method, reference, notes=""):
    new_row = {
        "System Name": system_name,
        "RA": ra,
        "Dec": dec,
        "Period": period,
        "Eccentricity": ecc,
        "M1": m1,
        "M1_sin3i": m1_sin3i,
        "M2": m2,
        "M2_sin3i": m2_sin3i,
        "q": q,
        "Mass Function": mass_func,
        "Type1": type1,
        "Type2": type2,
        "Detection Method": method,
        "Reference": reference,
        "Notes": notes,
    }
    return pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

TRI_NAN = [np.nan, np.nan, np.nan]

# ---- Parsing utilities (robust to odd TeX) ----
def clean_cell(s: str) -> str:
    s = s.strip()
    # join broken "NGC \\ \n 2420-173" cases in pre-pass
    return s

def to_float_or_nan(s: str) -> float | float:
    t = s.strip()
    t = t.replace(',', '')  # just in case
    # remove TeX wrappers
    t = re.sub(r"[${}\\]", "", t)
    t = t.replace(r"\,", "")
    t = t.replace(r"\cdot", "")
    t = t.replace("–", "-").replace("—", "-")
    # blanks or dashes
    if t == "" or t == "-" or t == "–":
        return np.nan
    # sometimes "0." appears; that's fine
    try:
        return float(t)
    except Exception:
        # pick the first number found (handles things like '0.' or '7E-05')
        m = re.search(r"[-+]?\d+(?:\.\d+)?(?:[Ee][-+]?\d+)?", t)
        return float(m.group(0)) if m else np.nan

def parse_asym_mass_triplet(s: str):
    """
    Parse M_Ba field like:
      '1.9$+0.7\\atop-0.5$'  → [0.5, 1.9, 0.7]
      '{\\sl 1.3}$+0.3\\atop-0.2$' → [0.2, 1.3, 0.3]
      '2.3$+1.4\\atop-0.7$' → [0.7, 2.3, 1.4]
      '1.6' → [nan, 1.6, nan]
    """
    t = s.strip()
    # keep a copy for fallback
    # normalize: turn "\atop" into a separator, strip TeX, keep signs
    t = t.replace("\\,", "")
    t = t.replace("\\atop", "|")
    t = re.sub(r"{\\sl\s*", "", t)       # drop slanted markup start
    t = t.replace("{", "").replace("}", "")
    t = t.replace("$", "")
    t = t.replace("\\", "")
    # extract numbers (value, +err, -err)
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?(?:[Ee][-+]?\d+)?", t)
    if len(nums) >= 3:
        val = float(nums[0])
        plus = abs(float(nums[1]))
        minus = abs(float(nums[2]))
        return [minus, val, plus]
    elif len(nums) >= 1:
        val = float(nums[0])
        return [np.nan, val, np.nan]
    else:
        return TRI_NAN.copy()

def as_triplet_val(x):
    """Value only → [nan, value, nan] triplet."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return TRI_NAN.copy()
    try:
        v = float(x)
        return [np.nan, v, np.nan]
    except Exception:
        return TRI_NAN.copy()

def cleanup_name(cell0: str) -> str:
    """
    Turn the 'HD/DM' cell into a clean System Name:
     - ' 5424'  → 'HD 5424'
     - '$-64^\\circ$4333' → 'DM -64 4333'
     - 'NGC 2420-173' → as-is
    """
    raw = cell0.strip()
    # join DM pattern like $-64^\circ$4333
    m = re.match(r"\s*\$?\s*([+\-]?\d+)\s*\^\s*\\?circ\s*\$?\s*(\d+)", raw)
    if m:
        deg, num = m.group(1), m.group(2)
        return f"DM {deg} {num}"
    # strip TeX
    s = re.sub(r"[${}\\]", "", raw)
    s = s.replace("^circ", "")
    s = s.replace("^\circ", "")
    s = re.sub(r"\s+", " ", s).strip()
    if s.upper().startswith("NGC"):
        return s.replace(" \\\\", "").strip()
    # pure number → HD
    if re.fullmatch(r"\d+", s):
        return f"HD {s}"
    return s

def parse_ba_class(cell: str) -> str:
    # e.g., 'm', 's', 'm$\rightarrow$s'
    t = cell.strip()
    t = t.replace("$", "").replace("\\rightarrow", "->")
    return re.sub(r"\s+", "", t)


# ---- Save HDF5 (triplets as datasets, rest as JSON metadata) ----
def save_triplet_columns_and_metadata(df, triplet_columns, h5_filename):
    with h5py.File(h5_filename, "w") as f:
        for col in triplet_columns:
            try:
                # Coerce to numeric triplets
                arr = np.array(df[col].tolist(), dtype=float)
                if arr.ndim != 2 or arr.shape[1] != 3:
                    raise ValueError(f"{col} is not triplets")
                f.create_dataset(col, data=arr)
            except Exception as e:
                print(f"Skipping column '{col}' due to: {e}")
        meta_cols = [c for c in df.columns if c not in triplet_columns]
        meta_json = df[meta_cols].to_json(orient="records")
        f.create_dataset("metadata_json", data=np.bytes_(meta_json))


#  Run the function with metadata saving
triplet_cols = ["RA", "Dec", "Period", "Eccentricity", "M1", "M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function",]




In [11]:

def symerr_to_triplet(val, err):
    """Convert (val, symmetric_err) -> [err, val, err] with NaNs handled."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return TRI_NAN.copy()
    if err is None or (isinstance(err, float) and np.isnan(err)):
        return [np.nan, float(val), np.nan]
    return [abs(float(err)), float(val), abs(float(err))]

def build_algols_malkov2020_h5(df_out_path):
    # VizieR config: get *all* rows, choose only columns we need
    Vizier.ROW_LIMIT = -1

    # Table metadata (from VizieR):
    # - GCVS : system name
    # - Per  : orbital period [days]
    # - M1, e_M1 ; M2, e_M2
    # - SpType
    # - BibCode (reference for that system’s parameters)
    # - _RA, _DE (coords from SIMBAD, injected by VizieR)
    # :contentReference[oaicite:2]{index=2}
    cols = ["GCVS", "Per", "M1", "e_M1", "M2", "e_M2", "SpType", "BibCode", "_RA", "_DE"]

    cat = "J/MNRAS/491/5489"
    tab = "tablea1"

    tables = Vizier(columns=cols).get_catalogs(f"{cat}/{tab}")
    if len(tables) == 0:
        raise RuntimeError("No tables returned from VizieR. Check catalog/table name.")
    t = tables[0].to_pandas()

    # Prepare your dataframe
    columns = [
        "System Name", "RA", "Dec", "Period", "Eccentricity",
        "M1","M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function",
        "Type1", "Type2", "Detection Method", "Reference", "Notes"
    ]
    df = pd.DataFrame(columns=columns)

    for _, row in t.iterrows():
        name = str(row["GCVS"]).strip() if pd.notna(row["GCVS"]) else ""
        if name == "" or name.lower() == "nan":
            # fallback: if any row lacks GCVS name, you can use another identifier,
            # but for this table GCVS is the primary.
            continue

        # Coordinates in degrees from SIMBAD
        ra_trip = as_triplet_val(row["_RA"])   # deg
        de_trip = as_triplet_val(row["_DE"])   # deg

        # Period (days)
        per_trip = as_triplet_val(row["Per"])

        # Eccentricity is NOT in this table → leave NaN triplet
        ecc_trip = TRI_NAN.copy()

        # Masses with symmetric errors
        m1_trip = symerr_to_triplet(row["M1"], row["e_M1"])
        m2_trip = symerr_to_triplet(row["M2"], row["e_M2"])

        # No sin^3 i products in this catalog (these are EB+SB2 physical masses)
        m1_sin3i = TRI_NAN.copy()
        m2_sin3i = TRI_NAN.copy()

        # Mass ratio
        if pd.notna(row["M1"]) and pd.notna(row["M2"]) and float(row["M1"]) != 0.0:
            q_val = float(row["M2"]) / float(row["M1"])
            q_trip = as_triplet_val(q_val)
        else:
            q_trip = TRI_NAN.copy()

        # Mass function not provided
        mf_trip = TRI_NAN.copy()

        # Types: you can keep spectral type as Type1 and leave Type2 blank,
        # or parse "B5+K3III" into two components.
        # Here: keep raw SpType in Type1 for now.
        sptype = str(row["SpType"]).strip() if pd.notna(row["SpType"]) else ""
        type1 = sptype
        type2 = ""

        # Detection method: this catalog is semidetached, *double-lined eclipsing binaries*
        method = "Eclipsing binary (phot) + SB2 (spec)"

        # Reference: per-row BibCode exists in the table
        ref = str(row["BibCode"]).strip() if pd.notna(row["BibCode"]) else "2020MNRAS.491.5489M"

        df = add_observation(
            df, system_name=name,
            ra=ra_trip, dec=de_trip, period=per_trip, ecc=ecc_trip,
            m1=m1_trip, m1_sin3i=m1_sin3i, m2=m2_trip, m2_sin3i=m2_sin3i,
            q=q_trip, mass_func=mf_trip,
            type1=type1, type2=type2,
            method=method, reference=ref,
            notes=""
        )

    # Save in your format
    triplet_cols = ["RA", "Dec", "Period", "Eccentricity", "M1", "M1_sin3i",
                    "M2", "M2_sin3i", "q", "Mass Function"]

    save_triplet_columns_and_metadata(df, triplet_cols, df_out_path)
    print(f"Built {len(df)} systems and saved to {df_out_path}")
    print(df[["System Name","Period","Eccentricity","M1","M2","q","Mass Function"]].head())

    return df



In [12]:
# Example usage:
from paths import DATA_DIR
out = DATA_DIR / "result_tables/Algols_Malkov2020.h5"
df = build_algols_malkov2020_h5(out)


Built 50 systems and saved to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/Algols_Malkov2020.h5
  System Name                  Period     Eccentricity  \
0      TW And  [nan, 4.12276035, nan]  [nan, nan, nan]   
1      WW And    [nan, 23.28525, nan]  [nan, nan, nan]   
2      RY Aqr   [nan, 1.9665826, nan]  [nan, nan, nan]   
3      XZ Aql    [nan, 2.139181, nan]  [nan, nan, nan]   
4      KO Aql    [nan, 2.864055, nan]  [nan, nan, nan]   

                                                  M1  \
0                      [nan, 1.684999942779541, nan]   
1  [0.09000000357627869, 3.180000066757202, 0.090...   
2  [0.07000000029802322, 1.2699999809265137, 0.07...   
3  [0.14000000059604645, 2.4200000762939453, 0.14...   
4  [0.05000000074505806, 2.5299999713897705, 0.05...   

                                                  M2  \
0                    [nan, 0.32499998807907104, nan]   
1  [0.09000000357627869, 0.5099999904632568, 0.09...   
2  [0.01999999955296516

### Inspect table before continuing

In [13]:
display(df)

# ## Turns out the HD/DM headers are accidentally taken in as data rows.
# ## Lets remove them.
# no_header_bool = df['System Name'] != 'HD/DM'

# df = df[no_header_bool].reset_index(drop=True)

# display(df)

,System Name,RA,Dec,Period,Eccentricity,M1,M1_sin3i,M2,M2_sin3i,q,Mass Function,Type1,Type2,Detection Method,Reference,Notes
0,TW And,"[nan, 0.82595, nan]","[nan, 32.84586, nan]","[nan, 4.12276035, nan]","[nan, nan, nan]","[nan, 1.684999942779541, nan]","[nan, nan, nan]","[nan, 0.32499998807907104, nan]","[nan, nan, nan]","[nan, 0.19287833775409974, nan]","[nan, nan, nan]",FV+KIV,,Eclipsing binary (phot) + SB2 (spec),2014AN....335.1064M,
1,WW And,"[nan, 356.22311, nan]","[nan, 45.68651, nan]","[nan, 23.28525, nan]","[nan, nan, nan]","[0.09000000357627869, 3.180000066757202, 0.090...","[nan, nan, nan]","[0.09000000357627869, 0.5099999904632568, 0.09...","[nan, nan, nan]","[nan, 0.16037735212481558, nan]","[nan, nan, nan]",A5+F3p,,Eclipsing binary (phot) + SB2 (spec),2012NewA...17..691S,
2,RY Aqr,"[nan, 320.06675, nan]","[nan, -10.80226, nan]","[nan, 1.9665826, nan]","[nan, nan, nan]","[0.07000000029802322, 1.2699999809265137, 0.07...","[nan, nan, nan]","[0.019999999552965164, 0.25999999046325684, 0....","[nan, nan, nan]","[nan, 0.2047244050142244, nan]","[nan, nan, nan]",A7+G8III,,Eclipsing binary (phot) + SB2 (spec),2016AJ....152...26M,
3,XZ Aql,"[nan, 305.55566, nan]","[nan, -7.35096, nan]","[nan, 2.139181, nan]","[nan, nan, nan]","[0.14000000059604645, 2.4200000762939453, 0.14...","[nan, nan, nan]","[0.05999999865889549, 0.44999998807907104, 0.0...","[nan, nan, nan]","[nan, 0.1859504024347856, nan]","[nan, nan, nan]",A2,,Eclipsing binary (phot) + SB2 (spec),2016AJ....152...33Z,
4,KO Aql,"[nan, 281.79476, nan]","[nan, 10.76368, nan]","[nan, 2.864055, nan]","[nan, nan, nan]","[0.05000000074505806, 2.5299999713897705, 0.05...","[nan, nan, nan]","[0.009999999776482582, 0.550000011920929, 0.00...","[nan, nan, nan]","[nan, 0.21739131151800167, nan]","[nan, nan, nan]",AV+[G8IV],,Eclipsing binary (phot) + SB2 (spec),2007MNRAS.379.1533S,
5,QS Aql,"[nan, 295.27303, nan]","[nan, 13.81568, nan]","[nan, 2.513294, nan]","[nan, nan, nan]","[0.09000000357627869, 4.070000171661377, 0.090...","[nan, nan, nan]","[0.05000000074505806, 1.4900000095367432, 0.05...","[nan, nan, nan]","[nan, 0.3660933529957381, nan]","[nan, nan, nan]",B5V+[A8IV],,Eclipsing binary (phot) + SB2 (spec),2017AJ....153...36Z,
6,V337 Aql,"[nan, 286.04247, nan]","[nan, -2.02974, nan]","[nan, 2.7338794, nan]","[nan, nan, nan]","[0.3100000023841858, 17.440000534057617, 0.310...","[nan, nan, nan]","[0.18000000715255737, 7.829999923706055, 0.180...","[nan, nan, nan]","[nan, 0.44896787178505404, nan]","[nan, nan, nan]",B0V+B3V,,Eclipsing binary (phot) + SB2 (spec),2014NewA...28...44T,
7,TT Aur,"[nan, 77.4262, nan]","[nan, 39.58632, nan]","[nan, 1.332735, nan]","[nan, nan, nan]","[nan, 8.100000381469727, nan]","[nan, nan, nan]","[nan, 5.400000095367432, nan]","[nan, nan, nan]","[nan, 0.6666666470437392, nan]","[nan, nan, nan]",B2V+B4,,Eclipsing binary (phot) + SB2 (spec),2004yCat.5115....0S,
8,IM Aur,"[nan, 78.87393, nan]","[nan, 46.40596, nan]","[nan, 1.24728912, nan]","[nan, nan, nan]","[nan, 2.380000114440918, nan]","[nan, nan, nan]","[nan, 0.7699999809265137, nan]","[nan, nan, nan]","[nan, 0.32352938819391325, nan]","[nan, nan, nan]",B7V+A5V,,Eclipsing binary (phot) + SB2 (spec),2004yCat.5115....0S,
9,IU Aur,"[nan, 81.96836, nan]","[nan, 34.78287, nan]","[nan, 1.8114743, nan]","[nan, nan, nan]","[0.07999999821186066, 11.989999771118164, 0.07...","[nan, nan, nan]","[0.03999999910593033, 6.070000171661377, 0.039...","[nan, nan, nan]","[nan, 0.5062552366583823, nan]","[nan, nan, nan]",O9.5V+B0.5IV-V,,Eclipsing binary (phot) + SB2 (spec),2009ASPC..404..178S,


## RA and DEC in the right format

In [6]:
# Collect the system names from the DataFrame
targets = df["System Name"].tolist()

display(df['System Name'].values)
targets = df['System Name'].values.tolist()

# Run the script and capture output
result = subprocess.run(["python3", "Get_Coords_From_SIMBAD.py"] + targets, capture_output=True, text=True)


# print("STDOUT:\n", result.stdout)
# Script prints one dict per system like:
#  *__89_HER = {
#     "System Name": '*  89 Her',
#     "RA":  [0.00001144, 268.854951, 0.00001144],
#     "Dec": [0.00001275, 26.049991, 0.00001275],
# }

# Get the printed output as a string
RA_DEC = result.stdout

entries = re.findall(r'([A-Z0-9_]+)\s*=\s*({.*?})', RA_DEC, re.DOTALL)
RADEC_data = {}

for name, dict_str in entries:
    RADEC_data[name] = ast.literal_eval(dict_str)

RADEC_data.keys()
# Now `RADEC_data` is a Python dictionary with the object data
# print(RADEC_data['HD__58978']['RA'])
# print(RADEC_data['HD__58978']['Dec'])

# Extract RA and Dec values
RA_values = [RADEC_data[key]['RA'] for key in RADEC_data.keys() ]
DEC_values = [RADEC_data[key]['Dec'] for key in RADEC_data.keys() ]


print(RA_values)
print(DEC_values)

df["RA"]  = RA_values
df["Dec"] = DEC_values   


array(['DM -64 4333', 'DM -42 2048', 'DM -14 2678', 'DM -01 3022',
       'HD 5424', 'HD 16458', 'HD 18182', 'HD 20394', 'HD 24035',
       'HD 27271', 'HD 31487', 'HD 40430', 'HD 43389', 'HD 44896',
       'HD 46407', 'HD 49641', 'HD 49841', 'HD 50082', 'HD 51959',
       'HD 53199', 'HD 58121', 'HD 58368', 'HD 59852', 'HD 77247',
       'HD 84678', 'HD 88562', 'HD 91208', 'HD 92626', 'HD 95193',
       'HD 98839', 'HD 101013', 'HD 104979', 'HD 107541', 'HD 119185',
       'HD 121447', 'HD 123949', 'HD 134698', 'HD 139195', 'HD 143899',
       'HD 154430', 'HD 178717', 'HD 180622', 'HD 183915', 'HD 196673',
       'HD 199939', 'HD 200063', 'HD 201657', 'HD 201824', 'HD 202109',
       'HD 204075', 'HD 205011', 'HD 210946', 'HD 211594', 'HD 218356',
       'HD 223617', 'NGC 2420-173'], dtype=object)

[[2.567e-05, 349.129716, 2.567e-05], [3.87e-06, 84.501334, 3.87e-06], [3.13e-06, 132.981509, 3.13e-06], [8.62e-06, 226.86194, 8.62e-06], [1.691e-05, 13.933035, 1.691e-05], [0.0001076, 41.948857, 0.0001076], [4.07e-06, 43.796224, 4.07e-06], [1.29e-05, 49.223252, 1.29e-05], [7.361e-05, 55.927469, 7.361e-05], [2.447e-05, 64.640935, 2.447e-05], [2.677e-05, 74.765533, 2.677e-05], [5.94e-06, 89.585, 5.94e-06], [1.084e-05, 93.946117, 1.084e-05], [1.815e-05, 95.724037, 1.815e-05], [4.275e-05, 98.195305, 4.275e-05], [1.086e-05, 102.372703, 1.086e-05], [2.367e-05, 102.662379, 2.367e-05], [2.673e-05, 102.974829, 2.673e-05], [5.26e-06, 104.792067, 5.26e-06], [5.08e-06, 106.201188, 5.08e-06], [1.503e-05, 111.11849, 1.503e-05], [2.298e-05, 111.412367, 2.298e-05], [5.63e-06, 112.94398, 5.63e-06], [7.31e-06, 135.884467, 7.31e-06], [3.938e-05, 145.46907, 3.938e-05], [1.82e-05, 153.124761, 1.82e-05], [1.563e-05, 157.888064, 1.563e-05], [3.227e-05, 160.208219, 3.227e-05], [1.882e-05, 164.861565, 1.882e-0

# Check what it looks like: 

In [7]:
display(df)

,System Name,RA,Dec,Period,Eccentricity,M1,M1_sin3i,M2,M2_sin3i,q,Mass Function,Type1,Type2,Detection Method,Reference,Notes
0,DM -64 4333,"[2.567e-05, 349.129716, 2.567e-05]","[1.178e-05, -63.451273, 1.178e-05]","[nan, 386.0, nan]","[nan, 0.03, nan]","[0.1, 1.4, 0.1]","[nan, nan, nan]","[nan, 0.61, nan]","[nan, nan, nan]","[nan, 0.4357142857142857, nan]","[nan, 0.068, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: s; [Fe/H]=-0.10; [hs/ls]=0.84; Type ...
1,DM -42 2048,"[3.87e-06, 84.501334, 3.87e-06]","[3.03e-06, -42.326459, 3.03e-06]","[nan, 3260.0, nan]","[nan, 0.08, nan]","[0.5, 1.9, 0.7]","[nan, nan, nan]","[nan, 0.74, nan]","[nan, nan, nan]","[nan, 0.3894736842105263, nan]","[nan, 0.065, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: s; [Fe/H]=-0.23; [hs/ls]=0.38; Type ...
2,DM -14 2678,"[3.13e-06, 132.981509, 3.13e-06]","[2.67e-06, -14.652608, 2.67e-06]","[nan, 3470.0, nan]","[nan, 0.22, nan]","[0.2, 3.0, 0.2]","[nan, nan, nan]","[nan, 0.8, nan]","[nan, nan, nan]","[nan, 0.26666666666666666, nan]","[nan, 0.023, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: m; [Fe/H]=0.01; [hs/ls]=0.04; Type m...
3,DM -01 3022,"[8.62e-06, 226.86194, 8.62e-06]","[7.56e-06, -2.305382, 7.56e-06]","[nan, 3253.0, nan]","[nan, 0.28, nan]","[0.1, 1.6, 0.1]","[nan, nan, nan]","[nan, 0.55, nan]","[nan, nan, nan]","[nan, 0.34375, nan]","[nan, 0.016, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: m; [Fe/H]=-0.14; [hs/ls]=-0.26; Type...
4,HD 5424,"[1.691e-05, 13.933035, 1.691e-05]","[1.397e-05, -27.893684, 1.397e-05]","[nan, 1881.0, nan]","[nan, 0.23, nan]","[0.3, 1.3, 0.4]","[nan, nan, nan]","[nan, 0.59, nan]","[nan, nan, nan]","[nan, 0.4538461538461538, nan]","[nan, 0.005, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: s; [Fe/H]=-0.43; [hs/ls]=0.57; Type ...
5,HD 16458,"[0.0001076, 41.948857, 0.0001076]","[1.678e-05, 81.448474, 1.678e-05]","[nan, 2018.0, nan]","[nan, 0.1, nan]","[0.1, 1.9, 0.1]","[nan, nan, nan]","[nan, 0.72, nan]","[nan, nan, nan]","[nan, 0.37894736842105264, nan]","[nan, 0.041, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: s; [Fe/H]=-0.64; [hs/ls]=0.19; Type ...
6,HD 18182,"[4.07e-06, 43.796224, 4.07e-06]","[3.94e-06, -4.316631, 3.94e-06]","[nan, 8059.0, nan]","[nan, 0.31, nan]","[0.1, 1.8, 0.2]","[nan, nan, nan]","[nan, 0.59, nan]","[nan, nan, nan]","[nan, 0.3277777777777778, nan]","[nan, 0.0002, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: m; [Fe/H]=-0.17; [hs/ls]=-0.05; Type...
7,HD 20394,"[1.29e-05, 49.223252, 1.29e-05]","[1.178e-05, 2.339352, 1.178e-05]","[nan, 2226.0, nan]","[nan, 0.2, nan]","[0.2, 2.0, 0.2]","[nan, nan, nan]","[nan, 0.76, nan]","[nan, nan, nan]","[nan, 0.38, nan]","[nan, 0.002, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: s; [Fe/H]=-0.27; [hs/ls]=0.42; Type ...
8,HD 24035,"[7.361e-05, 55.927469, 7.361e-05]","[2.25e-05, -72.609111, 2.25e-05]","[nan, 377.8, nan]","[nan, 0.3, nan]","[0.2, 1.3, 0.3]","[nan, nan, nan]","[nan, 0.57, nan]","[nan, nan, nan]","[nan, 0.4384615384615384, nan]","[nan, 0.047, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: s; [Fe/H]=-0.23; [hs/ls]=0.87; Type ...
9,HD 27271,"[2.447e-05, 64.640935, 2.447e-05]","[1.803e-05, 2.470529, 1.803e-05]","[nan, 1693.0, nan]","[nan, 0.22, nan]","[0.2, 2.9, 0.2]","[nan, nan, nan]","[nan, 0.79, nan]","[nan, nan, nan]","[nan, 0.2724137931034483, nan]","[nan, 0.024, nan]",barium star,white dwarf,[RV],[2019A&A...626A.127J],Ba class: m; [Fe/H]=-0.07; [hs/ls]=-0.13; Type...


## Finally convert and save your table to hdf5 format

In [ ]:

save_triplet_columns_and_metadata(df, triplet_cols, DATA_DIR / "result_tables/.h5")

print(f"Built {len(df)} systems and saved to barium_stars.h5")
print(df[["System Name","Period","Eccentricity","M1","M2","q","Mass Function"]].head())


Built 56 systems and saved to barium_stars.h5
   System Name              Period      Eccentricity               M1  \
0  DM -64 4333   [nan, 386.0, nan]  [nan, 0.03, nan]  [0.1, 1.4, 0.1]   
1  DM -42 2048  [nan, 3260.0, nan]  [nan, 0.08, nan]  [0.5, 1.9, 0.7]   
2  DM -14 2678  [nan, 3470.0, nan]  [nan, 0.22, nan]  [0.2, 3.0, 0.2]   
3  DM -01 3022  [nan, 3253.0, nan]  [nan, 0.28, nan]  [0.1, 1.6, 0.1]   
4      HD 5424  [nan, 1881.0, nan]  [nan, 0.23, nan]  [0.3, 1.3, 0.4]   

                 M2                                q      Mass Function  
0  [nan, 0.61, nan]   [nan, 0.4357142857142857, nan]  [nan, 0.068, nan]  
1  [nan, 0.74, nan]   [nan, 0.3894736842105263, nan]  [nan, 0.065, nan]  
2   [nan, 0.8, nan]  [nan, 0.26666666666666666, nan]  [nan, 0.023, nan]  
3  [nan, 0.55, nan]              [nan, 0.34375, nan]  [nan, 0.016, nan]  
4  [nan, 0.59, nan]   [nan, 0.4538461538461538, nan]  [nan, 0.005, nan]  
